In [1]:
from __future__ import annotations

from pathlib import Path
import re
from typing import Dict, Optional


def read_simulation_time(log_file: str | Path) -> Dict[str, Optional[float]]:
    """Read simulation timing information from an md-flexible logOutput file.

    Returns a dictionary with:
    - metric: Which line was parsed ("Total wall-clock time", "Simulate", or "Total accumulated")
    - nanoseconds: Parsed time in ns (float)
    - seconds: Parsed time in s (float)
    - file: Absolute path to the parsed file
    """
    path = Path(log_file).expanduser().resolve()
    text = path.read_text(encoding="utf-8", errors="replace")

    patterns = [
        r"Total wall-clock time\s*:\s*([0-9]+(?:\.[0-9]+)?)\s*ns\s*\(([0-9]+(?:\.[0-9]+)?)s\)",
        r"Simulate\s*:\s*([0-9]+(?:\.[0-9]+)?)\s*ns\s*\(([0-9]+(?:\.[0-9]+)?)s\)",
        r"Total accumulated\s*:\s*([0-9]+(?:\.[0-9]+)?)\s*ns\s*\(([0-9]+(?:\.[0-9]+)?)s\)",
    ]
    labels = ["Total wall-clock time", "Simulate", "Total accumulated"]

    for label, pattern in zip(labels, patterns):
        match = re.search(pattern, text)
        if match:
            return {
                "metric": label,
                "nanoseconds": float(match.group(1)),
                "seconds": float(match.group(2)),
                "file": str(path),
            }

    raise ValueError(
        f"No known simulation-time pattern found in {path}. Expected one of: "
        "'Total wall-clock time', 'Simulate', or 'Total accumulated'."
    )


# Example:
# result = read_simulation_time(
#     "generated_inputs_rangeCheck/totalParticles_300000/sigmaRatio_0p15/countRatio_2p00/dataLayout_AoS/run_0/logOutput_155157_2.out"
# )
# print(result)


In [2]:
import pandas as pd
import numpy as np

try:
    from scipy import stats as scipy_stats
    _HAS_SCIPY = True
except Exception:
    scipy_stats = None
    _HAS_SCIPY = False


def collect_runtime_data_rayleightaylor(base_dir: str | Path = ".", debug: bool = False) -> pd.DataFrame:
    """Collect simulation times from logOutput files for rayleighTaylor benchmark.
    
    Folder structure: traversal_<name>/cellSize_<value>/logOutput_*.out
    """
    base = Path(base_dir).expanduser().resolve()
    rows = []
    files_found = 0

    for log_path in base.rglob("logOutput_*.out"):
        files_found += 1
        # Extract traversal from parent's parent folder name (e.g., "traversal_hgrid_block4")
        traversal_folder = log_path.parent.parent.name
        if not traversal_folder.startswith("traversal_"):
            if debug:
                print(f"  Skipped: parent.parent.name={traversal_folder} (not traversal_*)")
            continue
        traversal = traversal_folder.replace("traversal_", "")
        
        # Extract cell_size from parent folder name (e.g., "cellSize_0p50")
        cellsize_folder = log_path.parent.name
        if not cellsize_folder.startswith("cellSize_"):
            if debug:
                print(f"  Skipped: parent.name={cellsize_folder} (not cellSize_*)")
            continue
        cell_size_str = cellsize_folder.replace("cellSize_", "")
        cell_size = float(cell_size_str.replace("p", "."))

        try:
            timing = read_simulation_time(log_path)
        except Exception as e:
            if debug:
                print(f"  Skipped: {log_path.name} (read error: {e})")
            continue
            
        rows.append(
            {
                "traversal": traversal,
                "cell_size": cell_size,
                "seconds": timing["seconds"],
                "log_file": str(log_path),
            }
        )
        if debug:
            print(f"  ✓ Loaded: {traversal:25} @ cellSize={cell_size:.2f} → {timing['seconds']:.3f}s")

    if debug:
        print(f"\nTotal logOutput files found: {files_found}")
        print(f"Successfully parsed: {len(rows)}")

    return pd.DataFrame(rows)


def t_critical(confidence: float, dof: int) -> float:
    """Two-sided t critical value for a given central confidence and dof."""
    if dof <= 0:
        return 0.0

    p = (1.0 + confidence) / 2.0

    if _HAS_SCIPY:
        return float(scipy_stats.t.ppf(p, dof))

    z_lookup = {
        0.68: 0.994,
        0.86: 1.080,
        0.90: 1.645,
        0.95: 1.960,
        0.99: 2.576,
    }
    return z_lookup.get(round(confidence, 2), 1.96)


def format_time_hhhmmss(seconds: float) -> str:
    """Format seconds into hh:mm:ss.ss format."""
    total_seconds = seconds
    hours = int(total_seconds // 3600)
    remaining = total_seconds % 3600
    minutes = int(remaining // 60)
    secs = remaining % 60
    return f"{hours:02d}:{minutes:02d}:{secs:05.2f}"


In [3]:

# Collect runtime data from the notebook's directory
import os
notebook_dir = Path(__file__).parent if "__file__" in dir() else Path.cwd()

runtime_df = collect_runtime_data_rayleightaylor(notebook_dir, debug=False)

if runtime_df.empty:
    print(f"No logOutput_*.out files found in {notebook_dir}")
else:
    print("Data collected:")
    print(f"  Traversals: {sorted(runtime_df['traversal'].unique().tolist())}")
    print(f"  Cell sizes: {sorted(runtime_df['cell_size'].unique().tolist())}")
    print(f"  Total measurements: {len(runtime_df)}")
    print()
    
    # Group by traversal, cell_size and compute statistics
    summary = runtime_df.groupby(["traversal", "cell_size"], as_index=False).agg(
        seconds_mean=("seconds", "mean"),
        n=("seconds", "count")
    ).sort_values(["traversal", "cell_size"])
    
    # Add column for runtime × 10,000 in hh:mm:ss.ss format
    summary["scaled_time_10k"] = (summary["seconds_mean"] * 10000).apply(format_time_hhhmmss)
    
    # Display results
    print("Runtime summary:")
    print("-" * 90)
    display_cols = ["traversal", "cell_size", "n", "seconds_mean", "scaled_time_10k"]
    result_table = summary[display_cols].copy()
    result_table.columns = ["Traversal", "Cell Size", "N", "Mean (s)", "Time × 10,000 (hh:mm:ss.ss)"]
    print(result_table.to_string(index=False))
    print("-" * 90)


Data collected:
  Traversals: ['hgrid_block4', 'hgrid_block8', 'hgrid_matching', 'lc_c08']
  Cell sizes: [0.5, 1.0]
  Total measurements: 8

Runtime summary:
------------------------------------------------------------------------------------------
     Traversal  Cell Size  N  Mean (s) Time × 10,000 (hh:mm:ss.ss)
  hgrid_block4        0.5  1 15245.317              42348:06:10.00
  hgrid_block4        1.0  1  6095.280              16931:20:00.00
  hgrid_block8        0.5  1 15895.094              44153:02:20.00
  hgrid_block8        1.0  1  6632.287              18423:01:10.00
hgrid_matching        0.5  1 19584.376              54401:02:40.00
hgrid_matching        1.0  1  5361.522              14893:07:00.00
        lc_c08        0.5  1  5444.820              15124:30:00.00
        lc_c08        1.0  1  5388.720              14968:40:00.00
------------------------------------------------------------------------------------------


## 1/2 In every direction and 1/2 number of iterations

In [6]:
# Display condensed summary table as DataFrame
if runtime_df.empty:
    print("runtime_df is empty. Run the previous cell first.")
else:
    # Build one row per traversal with requested time columns.
    summary_for_table = summary[["traversal", "cell_size", "seconds_mean"]].copy()

    pivot = summary_for_table.pivot_table(
        index="traversal",
        columns="cell_size",
        values="seconds_mean",
        aggfunc="mean"
    )

    final_summary = pivot.copy()
    final_summary["Fastest"] = final_summary.min(axis=1)
    final_summary = final_summary.sort_values("Fastest", ascending=True)

    # Reorder columns as requested.
    final_summary = final_summary.reindex(columns=["Fastest", 1.0, 0.5])
    final_summary = final_summary.rename(columns={1.0: "Cell Size 1", 0.5: "Cell Size 0.5"})

    def format_hhmmss(seconds: float) -> str:
        if pd.isna(seconds):
            return "-"
        total_seconds = int(round(float(seconds)))
        hours = total_seconds // 3600
        minutes = (total_seconds % 3600) // 60
        secs = total_seconds % 60
        return f"{hours:02d}:{minutes:02d}:{secs:02d}"

    for col in ["Fastest", "Cell Size 1", "Cell Size 0.5"]:
        if col in final_summary.columns:
            final_summary[col] = final_summary[col].apply(format_hhmmss)

    final_summary = final_summary.reset_index().rename(columns={"traversal": "Traversal"})

    print("\nFull Results Table:")
    print(final_summary.to_string(index=False))


Full Results Table:
     Traversal  Fastest Cell Size 1 Cell Size 0.5
hgrid_matching 01:29:22    01:29:22      05:26:24
        lc_c08 01:29:49    01:29:49      01:30:45
  hgrid_block4 01:41:35    01:41:35      04:14:05
  hgrid_block8 01:50:32    01:50:32      04:24:55


###  2/3 in each dimension and full 600 000 iterations###